In [9]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Set the HF_HOME environment variable
new_hf_home = "/home/ec2-user/SageMaker/decoding_exp/"
os.environ["HF_HOME"] = new_hf_home

print(os.environ["HF_HOME"])


/home/ec2-user/SageMaker/decoding_exp/


In [6]:
from huggingface_hub import notebook_login

# Replace with your Hugging Face token
hf_token = "
# Login to Hugging Face
notebook_login(hf_token)

In [ ]:
! pip install transformers -U

In [ ]:
! pip install bitsandbytes>=0.41.3

In [10]:
# load the tokenizer and model for our base model 
# load_in_4bit=True for resource management 
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1", torch_dtype=torch.float16, load_in_4bit=True, cache_dir = new_hf_home)
model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1",torch_dtype=torch.float16, load_in_4bit=True, cache_dir = new_hf_home)
model.eval()

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You are calling `save_pretrained` to a 4-bit converted model, but your `bitsandbytes` version doesn't support it. If you want to save 4-bit models, make sure to have `bitsandbytes>=0.41.3` installed.


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm()
        (post_attention_layernorm): MistralRMSNorm()
      )
    )

### LLMs are decoder-only models. The decoding for these models depends totlayy on model type. We either have causal LLM or Masked LM for the inference, which is pretrained and encoding properties are by default basic DPO/ a;lignment on top of sft techniques depending on task. 


In [ ]:
# basic decoding strategies covereed in this notebook,  and at the end misceelenous topics like latency, quantiization, watermarking config , if can be applied for safety tokens instead of greeen and red tokens 
# 1. greedy, 2. contrastive decoding, 3. beam search and its variations 4. pegasus diversity decoding(not relevant) 5. speculative for task type 6. chatLLM decoding which needs a prompt to perform

In [12]:
# 1 greedy 
# encode context the generation is conditioned on
input_ids = tokenizer.encode('I enjoy walking with my cute dog', return_tensors='pt')

# generate text until the output length (which includes the context length) reaches 50
greedy_output = model.generate(input_ids.to("cuda"), max_length=512)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I don’t like the fact that I have to clean up after him. I’m not a fan of the poop bags, either. I’m not sure why, but I just don’t like them.

I’ve been thinking about this for a while, and I’ve come up with a solution. I’m going to make a poop bag that I can use to pick up after my dog. I’m going to make it out of a material that is easy to clean and that will hold the poop.

I’m going to make it out of a material that is easy to clean and that will hold the poop.

I’m going to make it out of a material that is easy to clean and that will hold the poop.

I’m going to make it out of a material that is easy to clean and that will hold the poop.

I’m going to make it out of a material that is easy to clean and that will hold the poop.

I’m going to make it out of a material that is easy to clean and that will hold the poop.

I’m going to make

In [13]:
#2.  contrastive decoding,
prefix_text = r'I enjoy walking with my cute dog'
input_ids = tokenizer(prefix_text, return_tensors='pt').input_ids

# generate the result with contrastive search
output = model.generate(input_ids.to("cuda"), penalty_alpha=0.6, top_k=4, max_length=512)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(output[0], skip_special_tokens=True))
print("" + 100 * '-')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog.

It’s good for me to get out of the house and take a stroll around the neighbourhood.

I have a dog that is a great companion and a lot of fun to be around. He loves to go for walks and we have a great time when we’re out and about. The problem is that my dog doesn’t have a lot of manners. He pulls on the leash a lot and it’s hard to keep him under control. I tried training him, but he’s stubborn and won’t learn. I’m thinking about getting a harness for him so I can walk him without worrying about him choking himself. I know there are a lot of dog owners out there who have the same problem with their pups. I was looking at a website that specializes in dog harnesses and I found one that looked like it would work well for my dog. It has a handle on the back so I can hold on to him when he gets out of control. The other thing I like about it is th

In [14]:
# 3. beam search vanilla 


# activate beam search and early_stopping
beam_output = model.generate(
    input_ids.to("cuda"),  
    max_length=50, 
    num_beams=5, 
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to get him a dog treadmill.

Dog treadmills are great


In [16]:
# 4. beam search with n gram contraints

# set no_repeat_ngram_size to 2
beam_output = model.generate(
    input_ids.to("cuda"), 
    max_length=50, 
    num_beams=5, 
    no_repeat_ngram_size=2, 
    early_stopping=True
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to get him a dog treadmill.

I did some research and found the best


In [17]:
# 5. check various generated outputs in these. methods 
# set return_num_sequences > 1
beam_outputs = model.generate(
    input_ids.to("cuda"), 
    max_length=50, 
    num_beams=5, 
    no_repeat_ngram_size=2, 
    num_return_sequences=5, 
    early_stopping=True
)

# now we have 3 output sequences
print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
0: I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to get him a dog treadmill.

I did some research and found the best
1: I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to buy a dog treadmill for him.

I did a lot of research and
2: I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to buy a dog treadmill for him.

I did a lot of research before
3: I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to get him a dog treadmill.

I did a lot of research before buying
4: I enjoy walking with my cute dog, but I don’t always have the time to take him out for a long walk. So, I decided to get him a dog treadmill.

I did a lot of re

In [21]:
! pip install -q tensorflow

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
#5. top k sampling


import tensorflow as tf
# set seed to reproduce results. Feel free to change the seed though to get different results
tf.random.set_seed(0)

# set top_k to 50
sample_output = model.generate(
    input_ids.to('cuda'), 
    do_sample=True, 
    max_length=50, 
    top_k=50
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog in any weather but the cold winter months make me lazy in my leisurely rambles and my little friend is the same. She really enjoys going around the block when it’s raining or


In [22]:
# nucleus

# set seed to reproduce results. Feel free to change the seed though to get different results
tf.random.set_seed(0)

# deactivate top_k sampling and sample only from 92% most likely words
sample_output = model.generate(
    input_ids.to("cuda"), 
    do_sample=True, 
    max_length=50, 
    top_p=0.92, 
    top_k=0
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Output:
----------------------------------------------------------------------------------------------------
I enjoy walking with my cute dog in parks and forests, besides my regular jog. We are both particularly fond of our friendly neighborhood park. She eats her own poop; I clean hers. She licks me; I pet her.


In [25]:
# speculative decoding 
from transformers import AutoModelForCausalLM, AutoTokenizer

prompt = "How to make a bomb?"
checkpoint = "EleutherAI/pythia-1.4b-deduped"
assistant_checkpoint = "EleutherAI/pythia-160m-deduped"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
inputs = tokenizer(prompt, return_tensors="pt")

model = AutoModelForCausalLM.from_pretrained(checkpoint)
assistant_model = AutoModelForCausalLM.from_pretrained(assistant_checkpoint)
outputs = model.generate(**inputs, assistant_model=assistant_model, max_length = 50)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


['How to make a bomb?\n\nA:\n\nYou can use a simple timer to make']